# 4. Analysis Results & Visualizations

Produces inline visualizations **and** writes PNG / CSV artifacts to S3.

**Covers:**
1. Model performance — per-bin metrics table + PR / ROC curves
2. Feature importance — XGBoost gain (bar) + CatBoost FI side-by-side
3. SHAP global importance — bar chart + beeswarm (from SHAP parquet)
4. SHAP dependence plots — top 6 features
5. FFA / causal rules — top rules table + support vs confidence scatter
6. Cross-cohort feature overlap — falls vs ed top-50 feature Venn counts
7. Upload all artifacts to `s3://pgxdatalake/gold/analysis_visuals/`

**Prerequisites:** Steps 4–8 complete (model_events, final_model, SHAP, FFA).
Run from repo root.

In [ ]:
import os
import sys
import io
import warnings
from pathlib import Path

import boto3
import duckdb
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd

matplotlib.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 10,
})
warnings.filterwarnings('ignore')

def resolve_project_root() -> Path:
    cwd = Path.cwd()
    return cwd.parent if cwd.name in ('8_ffa_analysis', '7_shap_analysis', '6_final_model') else cwd

PROJECT_ROOT = resolve_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from py_helpers.constants import REQUIRED_COHORTS
from py_helpers.env_utils import get_data_root, get_model_data_root

S3_BUCKET   = os.environ.get('CPIC_S3_BUCKET', 'pgxdatalake')
AWS_PROFILE = os.environ.get('AWS_PROFILE')
DATA_ROOT   = get_data_root()
MODEL_DATA_ROOT = get_model_data_root()

FINAL_MODEL_OUT   = PROJECT_ROOT / '6_final_model' / 'outputs'
FINAL_MODEL_NVMe  = DATA_ROOT / '6_final_model' / 'outputs'
SHAP_OUT          = PROJECT_ROOT / '7_shap_analysis' / 'outputs'
FFA_OUT           = PROJECT_ROOT / '8_ffa_analysis' / 'outputs'
VISUALS_LOCAL     = PROJECT_ROOT / 'analysis_visuals'
VISUALS_LOCAL.mkdir(parents=True, exist_ok=True)

s3 = boto3.Session(profile_name=AWS_PROFILE).client('s3') if AWS_PROFILE else boto3.client('s3')

print(f'Project root : {PROJECT_ROOT}')
print(f'S3 bucket    : {S3_BUCKET}')
print(f'Cohorts      : {dict(REQUIRED_COHORTS)}')


In [ ]:
# ── Helpers ──────────────────────────────────────────────────────────────────

def abf(age_band: str) -> str:
    """65-74 → 65_74"""
    return age_band.replace('-', '_')

def model_base(cohort: str, age_band: str) -> Path:
    """Resolve 6_final_model outputs dir (project or NVMe)."""
    for base in (FINAL_MODEL_OUT, FINAL_MODEL_NVMe):
        d = base / cohort / abf(age_band)
        if d.exists():
            return d
    return FINAL_MODEL_OUT / cohort / abf(age_band)

def shap_base(cohort: str, age_band: str) -> Path:
    return SHAP_OUT / cohort / abf(age_band)

def ffa_base(cohort: str, age_band: str) -> Path:
    return FFA_OUT / cohort / abf(age_band)

def visuals_dir(cohort: str, age_band: str) -> Path:
    d = VISUALS_LOCAL / cohort / abf(age_band)
    d.mkdir(parents=True, exist_ok=True)
    return d

def save_and_upload(fig: plt.Figure, cohort: str, age_band: str, name: str) -> Path:
    """Save figure locally + upload PNG to S3 gold/analysis_visuals/."""
    local_path = visuals_dir(cohort, age_band) / f'{name}.png'
    fig.savefig(local_path, bbox_inches='tight')
    s3_key = f'gold/analysis_visuals/{cohort}/{abf(age_band)}/{name}.png'
    try:
        with open(local_path, 'rb') as fh:
            s3.upload_fileobj(fh, S3_BUCKET, s3_key,
                              ExtraArgs={'ContentType': 'image/png'})
        print(f'  [S3] s3://{S3_BUCKET}/{s3_key}')
    except Exception as exc:
        print(f'  [S3 WARN] {exc}')
    return local_path

def upload_csv(df: pd.DataFrame, cohort: str, age_band: str, name: str) -> None:
    """Upload CSV to S3 gold/analysis_visuals/."""
    local_path = visuals_dir(cohort, age_band) / f'{name}.csv'
    df.to_csv(local_path, index=False)
    s3_key = f'gold/analysis_visuals/{cohort}/{abf(age_band)}/{name}.csv'
    try:
        with open(local_path, 'rb') as fh:
            s3.upload_fileobj(fh, S3_BUCKET, s3_key,
                              ExtraArgs={'ContentType': 'text/csv'})
        print(f'  [S3] s3://{S3_BUCKET}/{s3_key}')
    except Exception as exc:
        print(f'  [S3 WARN] {exc}')

print('Helpers defined.')


## Section 1 — Model Performance
Per-bin metrics table + PR-AUC / Recall bar chart.

In [ ]:
from py_helpers.event_density_utils import DENSITY_BINS

all_metrics = []
for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        for bin_name in DENSITY_BINS:
            p = model_base(cohort, age_band) / 'bin_models' / bin_name \
                / f'{cohort}_{abf(age_band)}_model_metrics_summary.csv'
            if not p.exists():
                continue
            df = pd.read_csv(p)
            sel = df[df['selected'] == True] if 'selected' in df.columns else df.head(1)
            for _, row in sel.iterrows():
                all_metrics.append({
                    'cohort': cohort, 'age_band': age_band, 'bin': bin_name,
                    'model':       row.get('model', ''),
                    'recall_mean': round(float(row.get('recall_mean', 0)), 4),
                    'pr_auc_mean': round(float(row.get('pr_auc_mean', 0)), 4),
                    'auc_mean':    round(float(row.get('auc_mean', 0)), 4) if 'auc_mean' in row else None,
                    'logloss_mean':round(float(row.get('logloss_mean', 0)), 4) if 'logloss_mean' in row else None,
                })

if not all_metrics:
    print('[WARN] No model metrics found. Ensure Step 6 has run.')
else:
    metrics_df = pd.DataFrame(all_metrics)
    print('=== Model Performance Summary ===')
    display(metrics_df)

    # Upload summary CSV
    for cohort, bands in REQUIRED_COHORTS.items():
        sub = metrics_df[metrics_df['cohort'] == cohort]
        if not sub.empty:
            for age_band in bands:
                asub = sub[sub['age_band'] == age_band]
                if not asub.empty:
                    upload_csv(asub, cohort, age_band, 'model_metrics_summary')


In [ ]:
# Per-bin PR-AUC + Recall grouped bar chart per cohort/age_band
if all_metrics:
    for cohort, bands in REQUIRED_COHORTS.items():
        for age_band in bands:
            sub = metrics_df[(metrics_df['cohort'] == cohort) & (metrics_df['age_band'] == age_band)]
            if sub.empty:
                continue
            bins  = sub['bin'].tolist()
            x     = np.arange(len(bins))
            w     = 0.3
            fig, ax = plt.subplots(figsize=(max(8, len(bins) * 1.6), 4))
            ax.bar(x - w/2, sub['pr_auc_mean'],  w, label='PR-AUC',  color='#2196F3')
            ax.bar(x + w/2, sub['recall_mean'],  w, label='Recall',   color='#FF9800')
            ax.set_xticks(x)
            ax.set_xticklabels([f'{b}\n({r})' for b, r in zip(bins, sub['model'])], fontsize=8)
            ax.set_ylim(0, 1.05)
            ax.set_ylabel('Score')
            ax.set_title(f'Model Performance — {cohort} / {age_band}')
            ax.legend()
            ax.axhline(0.5, color='grey', lw=0.8, ls='--', alpha=0.5)
            plt.tight_layout()
            save_and_upload(fig, cohort, age_band, 'model_performance_by_bin')
            plt.show()


## Section 2 — Feature Importance (XGBoost Gain + CatBoost)

In [ ]:
for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        base = model_base(cohort, age_band)
        xgb_p = base / f'{cohort}_{abf(age_band)}_xgboost_feature_importance.csv'
        cat_p = base / f'{cohort}_{abf(age_band)}_catboost_feature_importance.csv'

        panels = []
        for path, label, color in [(xgb_p, 'XGBoost (gain)', '#1565C0'),
                                    (cat_p, 'CatBoost',       '#2E7D32')]:
            if path.exists():
                df = pd.read_csv(path).sort_values('importance', ascending=False).head(25)
                panels.append((df, label, color))

        if not panels:
            print(f'[skip] {cohort}/{age_band} — no FI CSVs')
            continue

        fig, axes = plt.subplots(1, len(panels), figsize=(9 * len(panels), 8))
        if len(panels) == 1:
            axes = [axes]
        for ax, (df, label, color) in zip(axes, panels):
            ax.barh(range(len(df)), df['importance'].values, color=color, alpha=0.85)
            ax.set_yticks(range(len(df)))
            ax.set_yticklabels(df['feature'].values, fontsize=8)
            ax.invert_yaxis()
            ax.set_xlabel('Importance')
            ax.set_title(f'{label}\n{cohort} / {age_band}')
        plt.tight_layout()
        save_and_upload(fig, cohort, age_band, 'feature_importance_top25')
        plt.show()


## Section 3 — SHAP Global Importance (Bar + Beeswarm)

In [ ]:
try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print('[WARN] shap not installed — beeswarm plots skipped. pip install shap')

sys.path.insert(0, str(PROJECT_ROOT / '8_ffa_analysis'))
try:
    from shap_parquet_loader import ShapParquetLoader
    LOADER_AVAILABLE = True
except ImportError:
    LOADER_AVAILABLE = False
    print('[WARN] ShapParquetLoader not importable — loading parquet with pandas fallback')

def load_shap_parquet(path: Path, top_n: int = 30) -> pd.DataFrame:
    """Load top_n features by mean |SHAP| from parquet (memory-efficient)."""
    if LOADER_AVAILABLE:
        loader = ShapParquetLoader(path)
        df = loader.to_pandas()
    else:
        df = pd.read_parquet(path)
    id_cols = {'mi_person_key', 'bias', 'instance_index'}
    feat_cols = [c for c in df.columns if c not in id_cols]
    mean_abs = df[feat_cols].abs().mean().sort_values(ascending=False)
    top_feats = mean_abs.head(top_n).index.tolist()
    return df[top_feats], mean_abs.head(top_n)

for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        sbase = shap_base(cohort, age_band)
        global_csv = sbase / f'{cohort}_{abf(age_band)}_shap_global_importance_xgboost.csv'
        sample_pq  = sbase / f'{cohort}_{abf(age_band)}_shap_sample_values_xgboost.parquet'

        # ── 3a: bar from global importance CSV ──
        if global_csv.exists():
            gi = pd.read_csv(global_csv).sort_values('mean_abs_shap', ascending=False).head(25)
            fig, ax = plt.subplots(figsize=(9, 7))
            ax.barh(range(len(gi)), gi['mean_abs_shap'].values, color='#7B1FA2', alpha=0.85)
            ax.set_yticks(range(len(gi)))
            ax.set_yticklabels(gi['feature'].values, fontsize=8)
            ax.invert_yaxis()
            ax.set_xlabel('Mean |SHAP|')
            ax.set_title(f'SHAP Global Importance (XGBoost)\n{cohort} / {age_band}')
            plt.tight_layout()
            save_and_upload(fig, cohort, age_band, 'shap_global_importance_bar')
            plt.show()
            upload_csv(gi, cohort, age_band, 'shap_global_importance_top25')
        else:
            print(f'[skip] {cohort}/{age_band} — no SHAP global importance CSV')

        # ── 3b: beeswarm from sample parquet ──
        if SHAP_AVAILABLE and sample_pq.exists():
            try:
                shap_df, _ = load_shap_parquet(sample_pq, top_n=20)
                shap_vals  = shap_df.values
                feat_names = shap_df.columns.tolist()
                # Need matching feature values — load model_events for same patients
                model_events_p = MODEL_DATA_ROOT / f'cohort_name={cohort}' / \
                                  f'age_band={age_band}' / 'model_events.parquet'
                if model_events_p.exists():
                    X_sample = duckdb.query(
                        f"SELECT {', '.join(feat_names)} "
                        f"FROM read_parquet('{model_events_p}') "
                        f"LIMIT {len(shap_df)}"
                    ).df()
                    X_sample = X_sample.reindex(columns=feat_names).fillna(0)
                    exp = shap.Explanation(
                        values=shap_vals,
                        data=X_sample.values,
                        feature_names=feat_names
                    )
                    fig, ax = plt.subplots(figsize=(10, 7))
                    shap.plots.beeswarm(exp, max_display=20, show=False)
                    ax = plt.gca()
                    ax.set_title(f'SHAP Beeswarm (XGBoost)\n{cohort} / {age_band}')
                    plt.tight_layout()
                    save_and_upload(plt.gcf(), cohort, age_band, 'shap_beeswarm')
                    plt.show()
                else:
                    print(f'[skip beeswarm] model_events not found for {cohort}/{age_band}')
            except Exception as exc:
                print(f'[WARN] Beeswarm failed for {cohort}/{age_band}: {exc}')
        elif not sample_pq.exists():
            print(f'[skip] {cohort}/{age_band} — no SHAP sample parquet')


## Section 4 — SHAP Dependence Plots (top 6 features)

In [ ]:
if SHAP_AVAILABLE:
    for cohort, bands in REQUIRED_COHORTS.items():
        for age_band in bands:
            sbase     = shap_base(cohort, age_band)
            sample_pq = sbase / f'{cohort}_{abf(age_band)}_shap_sample_values_xgboost.parquet'
            model_events_p = MODEL_DATA_ROOT / f'cohort_name={cohort}' / \
                              f'age_band={age_band}' / 'model_events.parquet'
            if not sample_pq.exists() or not model_events_p.exists():
                print(f'[skip] {cohort}/{age_band} — missing parquet for dependence plots')
                continue
            try:
                shap_df, mean_abs = load_shap_parquet(sample_pq, top_n=20)
                top6 = mean_abs.head(6).index.tolist()
                X_sample = duckdb.query(
                    f"SELECT {', '.join(top6)} "
                    f"FROM read_parquet('{model_events_p}') LIMIT {len(shap_df)}"
                ).df().reindex(columns=top6).fillna(0)

                fig, axes = plt.subplots(2, 3, figsize=(15, 8))
                axes = axes.flatten()
                for ax, feat in zip(axes, top6):
                    if feat not in shap_df.columns or feat not in X_sample.columns:
                        ax.set_visible(False)
                        continue
                    ax.scatter(
                        X_sample[feat].values,
                        shap_df[feat].values,
                        alpha=0.35, s=8, c='#7B1FA2'
                    )
                    ax.axhline(0, color='grey', lw=0.7, ls='--')
                    ax.set_xlabel(feat, fontsize=8)
                    ax.set_ylabel('SHAP value', fontsize=8)
                    ax.set_title(feat, fontsize=9)
                fig.suptitle(f'SHAP Dependence — Top 6 Features\n{cohort} / {age_band}',
                             fontsize=11)
                plt.tight_layout()
                save_and_upload(fig, cohort, age_band, 'shap_dependence_top6')
                plt.show()
            except Exception as exc:
                print(f'[WARN] Dependence plots failed for {cohort}/{age_band}: {exc}')
else:
    print('[skip] shap not available')


## Section 5 — FFA / Causal Rules

In [ ]:
FFA_RULE_GLOBS = [
    '*_top_rules*.csv',
    '*causal_rules*.csv',
    '*ffa_rules*.csv',
    '*rule_analysis*.csv',
]

for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        base = ffa_base(cohort, age_band)
        if not base.exists():
            print(f'[skip] {cohort}/{age_band} — no FFA outputs dir')
            continue

        rules_df = None
        for pattern in FFA_RULE_GLOBS:
            matches = sorted(base.glob(pattern))
            if matches:
                rules_df = pd.read_csv(matches[0])
                print(f'Loaded FFA rules: {matches[0].name}  ({len(rules_df)} rules)')
                break

        if rules_df is None:
            # Fallback: find any CSV in the FFA outputs
            csvs = sorted(base.glob('*.csv'))
            if csvs:
                rules_df = pd.read_csv(csvs[0])
                print(f'Loaded FFA CSV (fallback): {csvs[0].name}  ({len(rules_df)} rows)')
            else:
                print(f'[skip] {cohort}/{age_band} — no FFA CSVs found')
                continue

        top = rules_df.head(30)
        print(f'\nTop FFA rules — {cohort} / {age_band}')
        display(top)
        upload_csv(top, cohort, age_band, 'ffa_top_rules')

        # Support vs confidence scatter (if columns exist)
        has_support    = 'support'    in rules_df.columns
        has_confidence = 'confidence' in rules_df.columns
        has_lift       = 'lift'       in rules_df.columns
        if has_support and has_confidence:
            fig, ax = plt.subplots(figsize=(8, 5))
            sc = ax.scatter(
                rules_df['support'], rules_df['confidence'],
                c=rules_df['lift'] if has_lift else 'steelblue',
                cmap='viridis' if has_lift else None,
                alpha=0.65, s=25, edgecolors='none'
            )
            if has_lift:
                plt.colorbar(sc, ax=ax, label='Lift')
            ax.set_xlabel('Support')
            ax.set_ylabel('Confidence')
            ax.set_title(f'FFA Rules: Support vs Confidence\n{cohort} / {age_band}')
            plt.tight_layout()
            save_and_upload(fig, cohort, age_band, 'ffa_support_confidence_scatter')
            plt.show()


## Section 6 — Cross-Cohort Feature Overlap (falls vs ed)

In [ ]:
TOP_N_OVERLAP = 50

def top_features(cohort: str, age_band: str, n: int = TOP_N_OVERLAP) -> set:
    """Return top-n feature names from XGBoost FI CSV."""
    p = model_base(cohort, age_band) / f'{cohort}_{abf(age_band)}_xgboost_feature_importance.csv'
    if not p.exists():
        # Try SHAP global importance CSV
        p = shap_base(cohort, age_band) / f'{cohort}_{abf(age_band)}_shap_global_importance_xgboost.csv'
    if not p.exists():
        return set()
    df = pd.read_csv(p)
    val_col = 'importance' if 'importance' in df.columns else 'mean_abs_shap'
    return set(df.sort_values(val_col, ascending=False).head(n)['feature'].tolist())

cohorts = list(REQUIRED_COHORTS.keys())
all_bands = sorted({ab for bands in REQUIRED_COHORTS.values() for ab in bands})

for age_band in all_bands:
    sets = {c: top_features(c, age_band) for c in cohorts if age_band in REQUIRED_COHORTS.get(c, [])}
    if len(sets) < 2:
        continue

    cohort_list = list(sets.keys())
    a_feats, b_feats = sets[cohort_list[0]], sets[cohort_list[1]]
    if not a_feats or not b_feats:
        continue

    only_a   = a_feats - b_feats
    only_b   = b_feats - a_feats
    shared   = a_feats & b_feats

    # Overlap bar chart
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    labels  = [f'Only {cohort_list[0]}', 'Shared', f'Only {cohort_list[1]}']
    counts  = [len(only_a), len(shared), len(only_b)]
    colors  = ['#1565C0', '#43A047', '#E53935']
    axes[0].barh(labels, counts, color=colors, alpha=0.85)
    axes[0].set_xlabel('Feature count')
    axes[0].set_title(f'Top-{TOP_N_OVERLAP} Feature Overlap\n{age_band}')
    for i, v in enumerate(counts):
        axes[0].text(v + 0.3, i, str(v), va='center')

    # Shared feature list
    shared_list = sorted(shared)
    axes[1].axis('off')
    shared_text = '\n'.join(shared_list[:30]) + ('\n...' if len(shared_list) > 30 else '')
    axes[1].text(0.05, 0.95, f'Shared features ({len(shared_list)}):\n\n{shared_text}',
                 transform=axes[1].transAxes, va='top', fontsize=7.5,
                 family='monospace')
    plt.suptitle(f'Cross-cohort Feature Overlap — {age_band}', fontsize=11)
    plt.tight_layout()

    # Use first cohort dir for S3 upload of cross-cohort visuals
    save_and_upload(fig, cohort_list[0], age_band, f'cross_cohort_feature_overlap_{age_band}')
    plt.show()

    # Upload shared feature CSV
    overlap_df = pd.DataFrame({
        'feature': sorted(a_feats | b_feats),
        cohort_list[0]: [f in a_feats for f in sorted(a_feats | b_feats)],
        cohort_list[1]: [f in b_feats for f in sorted(a_feats | b_feats)],
        'shared': [f in shared for f in sorted(a_feats | b_feats)],
    })
    upload_csv(overlap_df, cohort_list[0], age_band, f'cross_cohort_feature_overlap_{age_band}')


## Section 7 — Upload Existing PNG Artifacts from Steps 6–8
Upload any PNGs already written by `run_shap_analysis.py` and `run_final_model.py`.

In [ ]:
uploaded = 0
skipped  = 0

SEARCH_DIRS = [
    ('6_final_model',  FINAL_MODEL_OUT),
    ('7_shap_analysis', SHAP_OUT),
    ('8_ffa_analysis',  FFA_OUT),
]

for step_name, base_dir in SEARCH_DIRS:
    if not base_dir.exists():
        continue
    for png in sorted(base_dir.rglob('*.png')):
        # Build S3 key mirroring the local structure under the step dir
        try:
            rel = png.relative_to(base_dir)
        except ValueError:
            rel = png.name
        s3_key = f'gold/analysis_visuals/{step_name}/{rel}'.replace('\\', '/')
        try:
            with open(png, 'rb') as fh:
                s3.upload_fileobj(fh, S3_BUCKET, s3_key,
                                  ExtraArgs={'ContentType': 'image/png'})
            uploaded += 1
        except Exception as exc:
            print(f'  [WARN] {png.name}: {exc}')
            skipped += 1

print(f'Uploaded {uploaded} PNG(s) from Steps 6–8; {skipped} skipped.')
print(f'S3 prefix: s3://{S3_BUCKET}/gold/analysis_visuals/')


## Section 8 — Artifact Inventory

In [ ]:
print(f'=== Local artifacts written by this notebook ===')
for f in sorted(VISUALS_LOCAL.rglob('*')):
    if f.is_file():
        size_kb = f.stat().st_size / 1024
        print(f'  {f.relative_to(VISUALS_LOCAL)}  ({size_kb:.1f} KB)')

print(f'\n=== S3 inventory: s3://{S3_BUCKET}/gold/analysis_visuals/ ===')
try:
    paginator = s3.get_paginator('list_objects_v2')
    for page in paginator.paginate(Bucket=S3_BUCKET, Prefix='gold/analysis_visuals/'):
        for obj in page.get('Contents', []):
            size_kb = obj['Size'] / 1024
            print(f"  s3://{S3_BUCKET}/{obj['Key']}  ({size_kb:.1f} KB)")
except Exception as exc:
    print(f'[WARN] S3 inventory failed: {exc}')
